# MiniMax-H3 on Amazon SageMaker

Deploy [MiniMaxAI/MiniMax-H3](https://huggingface.co/MiniMaxAI/MiniMax-H3) behind a SageMaker
**asynchronous** endpoint on one `ml.p5.48xlarge` (8x H100 80 GB) and drive every capability the open
checkpoints expose.

H3 is a joint audio-video diffusion transformer: one denoising loop over a single packed multimodal
sequence produces the video **and** its synchronised 32 kHz stereo soundtrack. No vocoder, no
separate audio pass. 24 fps, 4-15 seconds, 768-pixel short edge.

---

## How to use this notebook

**Run cell 1 (install) and cell 2 (bootstrap). After that, jump to any section you like.**

Every later cell depends on cell 2 and nothing else. That is deliberate: the endpoint costs
**$63.296/hour**, so re-running section 7 just to reach section 10 after a kernel restart is real
money. Two mechanisms make it work:

- **State lives in S3**, not in Python variables (`s3://<bucket>/minimax-h3/io/_notebook_state.json`).
  The role ARN, image URI, endpoint name and the S3 URI of every generated clip are recorded there.
  Restart the kernel, open a new Studio space, move machines — cell 2 picks it all back up.
- **Expensive steps are memoised.** Staging, the image build, the deploy and every demo check state
  first and skip if already done. Each says so. Pass `force=True` (or set `FORCE_REGEN = True`) when
  you actually want to redo one.

**Nothing is downloaded to the machine running this notebook.** Clips play from presigned S3 URLs, so
no MP4s accumulate locally and the `.ipynb` does not fill up with base64. First frames, last frames
and soundtracks are extracted **on the endpoint** (it already has ffmpeg) and written straight to S3,
which is what lets the keyframe and reference demos chain off a generated clip with zero local bytes.

### Section map

| Section | What it does | Cost | Idempotent |
|---|---|---|---|
| 1-2 | Install, bootstrap | free | yes |
| 3 | Preflight: quota, bucket, PassRole | free | yes |
| 4 | Dedicated least-privilege endpoint role | free | yes |
| 5 | Stage the 288 GB checkpoint to S3 | ~$0 | yes, resumable |
| 6 | Build and push the container image | ~$0 | skips if pushed |
| 7 | Deploy the endpoint | **$63.30/hr from here** | skips if InService |
| 8-12 | The demos | ~$1.06/min | cached per demo |
| 13 | Teardown | stops the meter | yes |

---

## What is open, and what is not

| Module | Role | Open |
|---|---|---|
| H3-Context-IR | Hosted rewriter producing the structured representation H3-Base was trained on | No |
| **H3-Base** | The 768p generator: `FL2VA` and `Ref2VA` partitions | **Yes — deployed here** |
| H3-Regenerate-2K | In-context regeneration at 2K | No |

The model card says Context-IR "is critical to the quality of the final output". `h3/prompts.py`
builds that structured prompt locally, following the two official writing guides. Section 12 shows
the hosted path if you have a MiniMax token.

## Three tasks, one endpoint

| Wire `task` | Inputs | Partition |
|---|---|---|
| `t2va` | text only | `FL2VA` |
| `fl2va` | text + first frame, last frame, or both | `FL2VA` |
| `ref2va` | text + <=9 images / <=3 videos / <=3 audio (12 total) | `Ref2VA` |

`Ref2VA` also covers video-to-video editing and lip-sync — reference *uses*, not extra task values.

The partitions are separate 144.1 GB checkpoints and the endpoint quota here is **one** p5, so
`container/serve.py` runs **two SGLang backends in one container** (GPUs 0-3 → `FL2VA`,
GPUs 4-7 → `Ref2VA`) and routes `/invocations` by task.

```
notebook ──put request JSON──▶ S3
    └──invoke_endpoint_async──▶ endpoint (1x ml.p5.48xlarge)
                                    │
                     ┌──────────────┴──────────────┐
                     │ container: serve.py (shim)   │
                     │  /ping   /invocations        │
                     │    ├─ sglang serve fl2va     │ GPUs 0-3  tp4  :30010
                     │    └─ sglang serve ref2va    │ GPUs 4-7  tp4  :30020
                     └──────────────┬──────────────┘
                    MP4 + PNG + WAV ─┴─▶ S3 ──▶ presigned URL ──▶ this notebook
```

---
## 1. Install

Local dependencies only. Nothing here is needed on the endpoint. Safe to re-run.

In [ ]:
%pip install -q --upgrade "boto3>=1.34" "huggingface_hub>=0.30"
print("done - restart the kernel only if pip actually upgraded something")

---
## 2. Bootstrap  ← **the one cell every other cell needs**

Cheap and idempotent: a few API calls and one S3 GET. Defines the configuration, the clients, the
S3-backed `state`, and the helpers the rest of the notebook uses.

### Set these

- `BUCKET` / `ENDPOINT_NAME` if you want names other than the defaults.
- `HF_TOKEN` — **optional**. MiniMax-H3 is *not* gated so staging works without one, but an
  authenticated Hub download gets higher rate limits and generally better CDN throughput on a
  288 GB pull. Paste a read token from <https://huggingface.co/settings/tokens>. It is used only by
  the staging step, on the machine running it, and is **never** put in the endpoint's environment
  (nothing to leak via `DescribeEndpointConfig`, and the container never talks to the Hub once
  weights are staged).

### Re-running this cell picks up edited helpers

It purges `h3*` from `sys.modules` before importing, so changing anything under `h3/` takes effect by
re-running this cell — no kernel restart. It also verifies that every module carries the symbols the
notebook needs and, if not, names the file to re-copy. A half-finished file sync is otherwise an
`ImportError` from thirty lines into this cell.

In [ ]:
# ============================ configuration =============================
REGION           = "us-east-1"
BUCKET           = None            # None -> sagemaker-<region>-<account>
ENDPOINT_NAME    = "minimax-h3-p5"
ENDPOINT_ROLE    = "minimax-h3-endpoint-role"
ECR_REPO         = "minimax-h3-sglang"
IMAGE_TAG        = "latest"

HF_TOKEN         = ""              # <-- optional: paste a Hugging Face read token here
                                   #     (faster/steadier 288 GB staging; H3 is not gated)

INSTANCE_TYPE    = "ml.p5.48xlarge"
VARIANTS         = "fl2va,ref2va"  # "fl2va" alone halves the weights and the cold start
GPUS_PER_BACKEND = 4
TP_SIZE          = 4               # 4x H100: tp4+ulysses1 = 49.8 GB/GPU, headroom for 15 s clips
ULYSSES_DEGREE   = 1               # tp2+ulysses2 is 4.6% faster but 66 GB/GPU
PERFORMANCE_MODE = "speed"         # speed | auto | memory

SNS_SUCCESS_TOPIC = ""             # optional SNS completion notifications
SNS_ERROR_TOPIC   = ""

FORCE_REGEN      = False           # True -> demos regenerate instead of reusing cached clips
# ========================================================================

import datetime as dt
import json
import os
import subprocess
import sys
import time
import urllib.parse
from pathlib import Path

import boto3
from IPython.display import HTML, display

# --- locate and (re)load the h3/ helper package ---------------------------
# Two hazards this handles, both of which produce confusing errors:
#
# 1. Jupyter caches imported modules in sys.modules, so editing h3/*.py has NO
#    effect until you restart the kernel. Purging the cache first means an edited
#    helper is picked up by re-running this cell.
# 2. A partial file sync (notebook copied, h3/ not) surfaces as
#    "ImportError: cannot import name X from 'h3'" from deep inside this cell.
#    The version + symbol check below names the stale files instead.
H3_HELPERS_REQUIRED = "2026.08.05.4"

PROJECT = Path.cwd()
if not (PROJECT / "h3" / "prompts.py").is_file():
    for cand in [*PROJECT.parents, PROJECT / "MinimaxH3", Path.home()]:
        if (cand / "h3" / "prompts.py").is_file():
            PROJECT = cand
            break
else:
    pass
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))

for _mod in [m for m in list(sys.modules) if m == "h3" or m.startswith("h3.")]:
    del sys.modules[_mod]

_NEEDED = {
    "h3.prompts": ["BasePrompt", "RefPrompt", "Shot", "dialogue", "voiceover"],
    "h3.client": ["H3AsyncEndpoint", "keyframe", "image_ref", "video_ref", "audio_ref"],
    "h3.deploy": ["EndpointSpec", "deploy", "wait_for_endpoint", "delete_endpoint"],
    "h3.iam": ["ensure_endpoint_role", "diagnose_endpoint_role", "endpoint_inline_policy"],
    "h3.codebuild": ["build_image"],
    "h3.state": ["State"],
}


def _check_helpers():
    import importlib

    stale = []
    for module_name, symbols in _NEEDED.items():
        path = PROJECT / Path(*module_name.split(".")).with_suffix(".py")
        if not path.is_file():
            stale.append(f"{path.relative_to(PROJECT)}  (missing)")
            continue
        try:
            mod = importlib.import_module(module_name)
        except Exception as exc:
            stale.append(f"{path.relative_to(PROJECT)}  (import failed: {exc})")
            continue
        absent = [s for s in symbols if not hasattr(mod, s)]
        if absent:
            stale.append(f"{path.relative_to(PROJECT)}  (out of date, missing {', '.join(absent)})")
    return stale


_stale = _check_helpers()
if _stale:
    raise ImportError(
        "The h3/ helper package under "
        + str(PROJECT)
        + " is incomplete or out of date:\n  "
        + "\n  ".join(_stale)
        + "\n\nRe-copy the whole h3/ directory (it is ~100 kB) plus scripts/ and container/"
          " from the project, then re-run this cell. No kernel restart needed."
    )

import h3 as _h3

if getattr(_h3, "__version__", "0") != H3_HELPERS_REQUIRED:
    print(f"WARNING h3 helpers are version {getattr(_h3, '__version__', 'unknown')}, "
          f"this notebook expects {H3_HELPERS_REQUIRED}.")
    print("        Re-copy h3/ if anything behaves oddly.")

from h3 import (
    BasePrompt, EndpointSpec, H3AsyncEndpoint, RefPrompt, Shot, State,
    audio_ref, build_image, delete_endpoint, delete_endpoint_role, deploy, dialogue,
    diagnose_endpoint_role,
    endpoint_inline_policy, ensure_endpoint_role, image_ref, keyframe,
    video_ref, voiceover, wait_for_endpoint,
)

# --- clients and derived names -------------------------------------------
session = boto3.Session(region_name=REGION)
sts, sm, s3, logs, iam, ecr, codebuild = (
    session.client(x)
    for x in ("sts", "sagemaker", "s3", "logs", "iam", "ecr", "codebuild")
)
ACCOUNT = sts.get_caller_identity()["Account"]
BUCKET = BUCKET or f"sagemaker-{REGION}-{ACCOUNT}"

WEIGHTS_KEY    = "minimax-h3/weights"
IO_KEY         = "minimax-h3/io"
WEIGHTS_PREFIX = f"s3://{BUCKET}/{WEIGHTS_KEY}"
IO_PREFIX      = f"s3://{BUCKET}/{IO_KEY}"
IMAGE_URI      = f"{ACCOUNT}.dkr.ecr.{REGION}.amazonaws.com/{ECR_REPO}:{IMAGE_TAG}"
LOG_GROUP      = f"/aws/sagemaker/Endpoints/{ENDPOINT_NAME}"

# --- S3-backed state: this is what makes any cell runnable on its own ----
state = State(s3, f"{IO_PREFIX}/_notebook_state.json")

# --- the endpoint client (constructing it calls nothing) -----------------
h3ep = H3AsyncEndpoint(
    endpoint_name=ENDPOINT_NAME,
    io_s3_prefix=IO_PREFIX,
    region_name=REGION,
    invocation_timeout_s=3600,
)


# ------------------------------- helpers --------------------------------
def run(cmd, env=None):
    # Run a subprocess, streaming its output into the notebook.
    print("$", " ".join(str(c) for c in cmd))
    merged = {**os.environ, **(env or {})}
    proc = subprocess.Popen([str(c) for c in cmd], stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1, env=merged)
    for line in proc.stdout:
        print(line, end="")
    return proc.wait()


def endpoint_status():
    try:
        d = sm.describe_endpoint(EndpointName=ENDPOINT_NAME)
        return d["EndpointStatus"], d.get("FailureReason")
    except Exception:
        return "NotFound", None


def require_endpoint():
    # Fail with a useful message rather than a cryptic 404 mid-demo.
    status, reason = endpoint_status()
    if status != "InService":
        raise RuntimeError(
            f"endpoint {ENDPOINT_NAME} is '{status}'"
            + (f" ({reason})" if reason else "")
            + ". Run section 7 (deploy) first, or wait for it to finish."
        )


def tail_logs(minutes=30, limit=60, pattern=""):
    kw = {"logGroupName": LOG_GROUP, "limit": limit,
          "startTime": int((time.time() - minutes * 60) * 1000)}
    if pattern:
        kw["filterPattern"] = pattern
    try:
        for e in logs.filter_log_events(**kw)["events"]:
            print(e["message"].rstrip())
    except logs.exceptions.ResourceNotFoundException:
        print(f"no log group {LOG_GROUP} yet - the container has not started")


def s3_size(key_prefix):
    # (objects, bytes) under a key prefix.
    n = total = 0
    for page in s3.get_paginator("list_objects_v2").paginate(Bucket=BUCKET, Prefix=key_prefix):
        for o in page.get("Contents", []):
            n += 1
            total += o["Size"]
    return n, total


def presign(uri, expires_s=6 * 3600):
    p = urllib.parse.urlparse(uri)
    return s3.generate_presigned_url(
        "get_object", Params={"Bucket": p.netloc, "Key": p.path.lstrip("/")}, ExpiresIn=expires_s
    )


def show_image(uri, width=320, label=""):
    # Display an S3 image without downloading it.
    display(HTML(
        f"<p style='font:600 12px sans-serif;margin:6px 0 2px'>{label or uri.rsplit('/', 1)[-1]}</p>"
        f'<img src="{presign(uri)}" width="{width}">'
    ))


def demo(name, *, force=False, **kwargs):
    # Generate one demo clip, or reuse the one already in S3.
    #
    # This is what makes the demo cells re-runnable: the clip is recorded in
    # state, so a second execution costs a dictionary lookup instead of a
    # minutes-long p5 generation. Pass force=True (or set FORCE_REGEN) to redo it.
    require_endpoint()
    key = f"demo.{name}"
    rec = state.cached(
        key,
        lambda: h3ep.generate(name=name, **kwargs).as_state(),
        force=force or FORCE_REGEN,
    )
    if rec.get("summary"):
        print(rec["summary"])
    return h3ep.rehydrate(rec)


# ---------------------- shared fixtures ---------------------------------
# Defined here, not in the cell that first uses them, so that any later cell
# works after running only this bootstrap.

STAGE_CMD = [sys.executable, str(PROJECT / "scripts" / "stage_weights.py"),
             "--bucket", BUCKET, "--prefix", WEIGHTS_KEY, "--variants", VARIANTS]
STAGE_ENV = {"HF_TOKEN": HF_TOKEN} if HF_TOKEN else {}


def have_docker():
    return subprocess.run(["docker", "info"], capture_output=True).returncode == 0


def make_spec():
    # Built lazily: it reads the image URI and role ARN out of state, which do
    # not exist until sections 4 and 6 have run.
    return EndpointSpec(
        name=ENDPOINT_NAME,
        image_uri=state.require("image.pushed", hint="section 6")["image_uri"],
        role_arn=state.require("role.arn", hint="section 4"),
        async_output_s3_prefix=IO_PREFIX,
        model_data_s3_prefix=WEIGHTS_PREFIX,
        generated_s3_prefix=f"{IO_PREFIX}/generated",
        instance_type=INSTANCE_TYPE,
        variants=VARIANTS,
        gpus_per_backend=GPUS_PER_BACKEND,
        tp_size=TP_SIZE,
        ulysses_degree=ULYSSES_DEGREE,
        performance_mode=PERFORMANCE_MODE,
        warmup_on_start=True,
        sns_success_topic_arn=SNS_SUCCESS_TOPIC or None,
        sns_error_topic_arn=SNS_ERROR_TOPIC or None,
        # SGLang rejects any warmup frame that is not H3's native 768 short edge.
        extra_env={"H3_WARMUP_SHORT_EDGE": "768"},
    )


# The prompt shared by the knob demos in section 11.
TIDE_SHOT = Shot(
    "Cinematic, live-action, a static shot frames a shallow tide pool on dark volcanic rock at dusk. "
    "Water ripples across the pool and a small red anemone opens and closes at its centre. Thin "
    "spray drifts in from the left edge of the frame."
)
TIDE_SOUND = (
    "Waves break at a steady distance with a low continuous rush. Water laps close to the microphone "
    "and a faint wind moves across the rock."
)


def tide(duration):
    return BasePrompt(mode="t2va", duration_s=duration, shots=[TIDE_SHOT],
                      soundscape=TIDE_SOUND, music="N/A")


def artifact(demo_name, which):
    # S3 URI of a first_frame / last_frame / audio extracted from a demo clip.
    rec = state.require(f"demo.{demo_name}",
                        hint=f"the '{demo_name}' demo cell (it produces this artifact)")
    arts = rec.get("artifacts") or {}
    if which not in arts:
        raise KeyError(
            f"demo '{demo_name}' has no '{which}' artifact. Regenerate it with extract=True: "
            f"demo('{demo_name}', force=True, ...)"
        )
    return arts[which]


print(f"helpers   h3 {_h3.__version__} from {PROJECT}")
print(f"account   {ACCOUNT}   region {REGION}")
print(f"bucket    s3://{BUCKET}")
print(f"weights   {WEIGHTS_PREFIX}")
print(f"io        {IO_PREFIX}")
print(f"image     {IMAGE_URI}")
print(f"endpoint  {ENDPOINT_NAME} ({INSTANCE_TYPE}, variants={VARIANTS})  status: {endpoint_status()[0]}")
print(f"hf token  {'set' if HF_TOKEN else 'not set (fine - H3 is not gated)'}")
print()
print(state.summary())

### Reading the state

`state.summary()` above tells you what has already been done, so you know which sections you can
skip. Useful one-liners:

```python
state.keys()                  # everything recorded
state.get("image.pushed")     # None if section 6 has not run
state.drop("demo.t2va")       # forget one demo so it regenerates
```

---
## 3. Preflight

Read-only apart from hardening the bucket if it has to be created. Idempotent.

In [ ]:
ok = True

# --- quota ---------------------------------------------------------------
want = f"{INSTANCE_TYPE} for endpoint usage"
found = None
for page in session.client("service-quotas").get_paginator("list_service_quotas").paginate(
        ServiceCode="sagemaker"):
    for q in page["Quotas"]:
        if q["QuotaName"] == want:
            found = q["Value"]
print(f"quota  {want}: {found}")
if not found or found < 1:
    ok = False
    print("  -> request an increase before section 7")

# --- bucket: create locked down if absent -------------------------------
try:
    s3.head_bucket(Bucket=BUCKET)
    print(f"bucket s3://{BUCKET} reachable")
except Exception:
    kw = {"Bucket": BUCKET}
    if REGION != "us-east-1":
        kw["CreateBucketConfiguration"] = {"LocationConstraint": REGION}
    s3.create_bucket(**kw)
    s3.put_public_access_block(Bucket=BUCKET, PublicAccessBlockConfiguration={
        "BlockPublicAcls": True, "IgnorePublicAcls": True,
        "BlockPublicPolicy": True, "RestrictPublicBuckets": True})
    s3.put_bucket_encryption(Bucket=BUCKET, ServerSideEncryptionConfiguration={"Rules": [
        {"ApplyServerSideEncryptionByDefault": {"SSEAlgorithm": "AES256"}, "BucketKeyEnabled": True}]})
    print(f"bucket s3://{BUCKET} created with Block Public Access + SSE-S3")

try:
    pab = s3.get_public_access_block(Bucket=BUCKET)["PublicAccessBlockConfiguration"]
    if not all(pab.values()):
        print(f"  WARNING: Block Public Access not fully enabled: {pab}")
except Exception:
    print("  WARNING: no Block Public Access configuration on this bucket - review it")

# --- iam:PassRole -------------------------------------------------------
# create_model hands ENDPOINT_ROLE to SageMaker. Without PassRole you get an
# AccessDenied that never mentions PassRole.
caller = sts.get_caller_identity()["Arn"]
principal = caller
if ":assumed-role/" in caller:
    principal = f"arn:aws:iam::{ACCOUNT}:role/{caller.split('/')[1]}"
# Two groups: deploying the endpoint, and building the image in CodeBuild.
# iam:PassRole appears in both - create_model passes the endpoint role to
# SageMaker, create_project passes the build role to CodeBuild - and in both
# cases the AccessDenied you get without it never mentions PassRole.
probes = [
    (["iam:PassRole", "iam:CreateRole", "iam:PutRolePolicy"],
     [f"arn:aws:iam::{ACCOUNT}:role/{ENDPOINT_ROLE}",
      f"arn:aws:iam::{ACCOUNT}:role/minimax-h3-codebuild-role"]),
    (["sagemaker:CreateModel", "sagemaker:CreateEndpoint"], ["*"]),
    (["codebuild:CreateProject", "codebuild:StartBuild", "codebuild:BatchGetBuilds"], ["*"]),
]
for actions, resources in probes:
    try:
        sim = iam.simulate_principal_policy(
            PolicySourceArn=principal, ActionNames=actions, ResourceArns=resources)
        for r in sim["EvaluationResults"]:
            decision = r["EvalDecision"]
            print(f"perm   {r['EvalActionName']:<32} {decision}"
                  + ("" if decision == "allowed" else "   <-- needed"))
    except Exception as exc:
        print(f"perm   could not simulate {actions[0]}... ({type(exc).__name__}) - check by hand")

print()
print("PREFLIGHT OK" if ok else "PREFLIGHT INCOMPLETE - fix the above")

---
## 4. Dedicated endpoint role

The reflex is `sagemaker.get_execution_role()` plus `AmazonSageMakerFullAccess`. Don't — that role
can create endpoints and read the whole default bucket. `whisper_yd`'s steering in this workspace is
explicit: *do not revert the endpoint to the broad SageMaker notebook role*. So the endpoint gets its
own role with eight narrow statements:

| Statement | Why |
|---|---|
| `PullOurImageLayers` | our ECR repo only — this is a BYOC image, no AWS DLC access needed |
| `EcrAuthToken` | `GetAuthorizationToken` has no resource-level permissions; `"*"` is the only legal resource |
| `ReadStagedWeights` | `GetObject` on the weights prefix (SageMaker downloads 288 GB as this role) |
| `ListOnlyOurPrefixes` | `ListBucket` conditioned on `s3:prefix` — an uncompressed prefix must be enumerated |
| `ReadRequestsAndConditions` | `GetObject` on `requests/`, `demo-inputs/`, `generated/` — the demo chain feeds a clip's own extracted frames back in as conditions |
| `WriteOutputs` | `PutObject` on `generated/`, `responses/`, `failures/`. No `DeleteObject` |
| `OwnLogGroupOnly` | logs scoped to this endpoint's group, not `*` |
| `NamespacedMetrics` | `PutMetricData` conditioned on the SageMaker namespaces |

Trust is `sagemaker.amazonaws.com` with `aws:SourceAccount`, closing the confused-deputy hole.
Idempotent — re-running updates the policy in place.

### What *you* need (as opposed to the endpoint)

The deploying principal needs rather more than the endpoint does:

- `sagemaker:CreateModel` / `CreateEndpointConfig` / `CreateEndpoint` / the matching `Delete*`
- `iam:CreateRole` + `iam:PutRolePolicy` for this cell, and for the CodeBuild role in section 6
- **`iam:PassRole`** on *both* roles. `create_model` passes the endpoint role to SageMaker and
  `create_project` passes the build role to CodeBuild; in both cases the `AccessDenied` you get
  without it never mentions PassRole.
- `codebuild:CreateProject` / `UpdateProject` / `StartBuild` / `BatchGetBuilds` if you build in
  CodeBuild, which you probably will (section 6).

Section 3 simulates all of these, so a missing one shows up before it costs you a 40-minute deploy.

### Why none of these statements uses a `Condition`

An earlier version scoped `s3:ListBucket` with a `Condition` on `s3:prefix`, and conditioned the trust
policy on `aws:SourceAccount`. Both looked tighter. Both made the deploy fail:

```
The provided role 'arn:aws:iam::...:role/minimax-h3-endpoint-role' is invalid. Please ensure
the role exists, that its trust relationship policy allows the action "sts:AssumeRole" for the
service principal "sagemaker.amazonaws.com" and has "s3:ListBucket", and "s3:PutObject"
permissions for bucket '...'.
```

SageMaker validates the execution role when you create the model, and it evaluates the policy with an
**empty request context**. A `Condition` that references a request key — `s3:prefix`,
`aws:SourceAccount` — evaluates *false* there, even though it would pass in real use. So the
least-privilege mechanism that reads best is the one that breaks role validation.

The lesson generalises: scope a service role by **resource ARN**, not by `Condition`. The policy above
does that, and `diagnose_endpoint_role()` in the next cell reproduces each check SageMaker makes so
you can see it rather than guess from that message, which names three things without saying which
failed.

In [ ]:
ROLE_ARN = state.cached("role.arn", lambda: ensure_endpoint_role(
    iam,
    role_name=ENDPOINT_ROLE,
    account_id=ACCOUNT,
    region=REGION,
    bucket=BUCKET,
    weights_prefix=WEIGHTS_KEY,
    io_prefix=IO_KEY,
    ecr_repo=ECR_REPO,
    endpoint_name=ENDPOINT_NAME,
))
print("role:", ROLE_ARN)

In [ ]:
# Would SageMaker accept this role? Simulates each check in its error message.
diagnose_endpoint_role(
    iam,
    role_name=ENDPOINT_ROLE,
    account_id=ACCOUNT,
    bucket=BUCKET,
    weights_prefix=WEIGHTS_KEY,
    io_prefix=IO_KEY,
    ecr_repo=ECR_REPO,
    region=REGION,
)

In [ ]:
# Repair the role in place: rewrites both the trust policy and the inline policy,
# then waits for IAM to settle. Run this if the deploy in section 7 said
#   "The provided role '...' is invalid"
# or if you changed h3/iam.py. state.cached() would otherwise skip the cell above.
ROLE_ARN = ensure_endpoint_role(
    iam, role_name=ENDPOINT_ROLE, account_id=ACCOUNT, region=REGION, bucket=BUCKET,
    weights_prefix=WEIGHTS_KEY, io_prefix=IO_KEY, ecr_repo=ECR_REPO,
    endpoint_name=ENDPOINT_NAME,
    restrict_trust_to_account=False,   # True adds aws:SourceAccount and BREAKS validation
)
state.put("role.arn", ROLE_ARN)

print()
diagnose_endpoint_role(
    iam, role_name=ENDPOINT_ROLE, account_id=ACCOUNT, bucket=BUCKET,
    weights_prefix=WEIGHTS_KEY, io_prefix=IO_KEY, ecr_repo=ECR_REPO, region=REGION)
print()
print(json.dumps(endpoint_inline_policy(
    account_id=ACCOUNT, region=REGION, bucket=BUCKET, weights_prefix=WEIGHTS_KEY,
    io_prefix=IO_KEY, ecr_repo=ECR_REPO, endpoint_name=ENDPOINT_NAME), indent=1))

> **Rotate any credentials you pasted into a chat, ticket or other shared context.** This workspace
> has been bitten by that before (`whisper_yd/SKILL.md`, pitfall 16). Prefer the execution role or a
> scoped profile; never persist session tokens to disk.

---
## 5. Stage the checkpoint to S3

The two SGLang partitions are 144.1 GB each — **288 GB**. `scripts/stage_weights.py` relays each file
**Hub → S3 through a bounded in-memory buffer**: nothing is written to local disk, so the machine
running this needs no scratch space and its EBS throughput is not the bottleneck.

It also records the repo revision in `_h3_stage.json`. The container turns the staged tree into a warm
Hugging Face cache entry at that exact revision (symlinks, no copy), which is what lets
`sglang serve --model-path MiniMaxAI/MiniMax-H3` resolve local weights offline. The cookbook is
explicit that SGLang owns the checkpoint-directory mapping and you should *not* point `--model-path`
at a hand-downloaded subdirectory.

**Resumable.** A file already in S3 at the right size is skipped, and the script verifies every object
against the manifest at the end. If it dies halfway, just run it again.

Expect 30-120 minutes. Long enough that a browser disconnect is a real risk — the next-but-one cell
prints the exact terminal command if you would rather run it there.

In [ ]:
# Where do we stand? Cheap, read-only.
n, total = s3_size(WEIGHTS_KEY + "/")
print(f"staged: {n} objects, {total / 1e9:.1f} GB under {WEIGHTS_PREFIX}")
try:
    marker = json.loads(s3.get_object(Bucket=BUCKET, Key=f"{WEIGHTS_KEY}/_h3_stage.json")["Body"].read())
    print("complete:", json.dumps(marker, indent=1))
    state.put("weights.staged", marker)
except Exception:
    print("_h3_stage.json absent -> staging has not completed. Run the next cell.")

print("\nto run it in a terminal instead:")
print("  " + " ".join(STAGE_CMD) + " --workers 6")

In [ ]:
# Dry run: exactly what would move, and what is already there.
run(STAGE_CMD + ["--dry-run"], env=STAGE_ENV)

In [ ]:
# The real thing, 288 GB. Skips anything already staged.
def _stage():
    rc = run(STAGE_CMD + ["--workers", "6"], env=STAGE_ENV)
    if rc != 0:
        raise RuntimeError("staging failed - see the output above; re-run to retry what is missing")
    return json.loads(s3.get_object(Bucket=BUCKET, Key=f"{WEIGHTS_KEY}/_h3_stage.json")["Body"].read())

marker = state.cached("weights.staged", _stage)
print(json.dumps(marker, indent=1))

---
## 6. Build and push the image

`container/Dockerfile` starts from `lmsysorg/sglang:dev` (matching torch/CUDA plus the compiled
`sgl-kernel` and flashinfer stack), moves the bundled SGLang source to a pinned `main` commit that
carries the MiniMax-H3 pipeline, and installs the `[diffusion]` extra. MiniMax-H3 is **not in any
tagged SGLang release** — verified absent from v0.5.16, present in `main`.

Two details worth knowing:

- It **asserts at build time** that `sglang.multimodal_gen...minimax_h3` imports. A missing pipeline
  should fail a 20-minute build, not a 40-minute deploy.
- It clears `ENTRYPOINT` and sets `CMD ["serve"]`, because SageMaker hosting literally runs
  `docker run <image> serve`.

### Where to build: almost certainly CodeBuild

The base image is 13.7 GB compressed and **~30 GB extracted**, which is more than either obvious
host can take:

| Host | Docker? | Problem |
|---|---|---|
| SageMaker **Studio** | no daemon unless the domain sets `EnableDockerAccess` | and a space's EBS defaults to 100 GB max |
| SageMaker **notebook instance** | yes | Docker's data-root is on the *fixed* ~100 GB root volume, typically 10-20 GB free. `VolumeSizeInGB` does **not** size that volume |
| **CodeBuild** | yes | 128 GB of build disk, nothing to configure |

So the next two cells do it in CodeBuild: zip `container/`, create a least-privilege build project
(`BUILD_GENERAL1_LARGE`, privileged mode for Docker-in-Docker), run it, stream the logs, push to ECR.
`BUILD_GENERAL1_SMALL` would be the default elsewhere and its 64 GB is too tight here.

If you *do* have a working local daemon with ~80 GB free on Docker's data-root, the third cell builds
locally instead — and `scripts/notebook-lifecycle-on-start.sh` relocates the data-root onto the ML
volume for a notebook instance.

In [ ]:
# Which build path applies here?
import shutil

for p in ("/", "/home/ec2-user/SageMaker", "/home/sagemaker-user", "/tmp"):
    try:
        u = shutil.disk_usage(p)
        print(f"{p:<28} {u.free / 1e9:6.1f} GB free / {u.total / 1e9:6.1f} GB")
    except OSError:
        pass

probe = subprocess.run(["docker", "info", "--format", "{{.DockerRootDir}}"],
                       capture_output=True, text=True)
if probe.returncode != 0:
    print("\nNo Docker daemon here (normal in SageMaker Studio).")
    print("-> use the CodeBuild cell below. Nothing else to set up.")
else:
    root = probe.stdout.strip()
    free = shutil.disk_usage(root).free / 1e9
    print(f"\nDocker data-root {root}  ({free:.1f} GB free)")
    if free < 80:
        print("  Not enough: the base image alone extracts to ~30 GB, the build wants ~80 GB.")
        print("  -> use the CodeBuild cell below, or relocate the data-root with")
        print("     scripts/notebook-lifecycle-on-start.sh and use the local-build cell.")
    else:
        print("  Enough room - either the CodeBuild cell or the local-build cell will work.")

In [ ]:
# Build in CodeBuild. This is the path that works on Studio and on a notebook
# instance. 20-40 min the first time; skipped entirely if already recorded.
#
# If it fails on disk, re-run with compute_type="BUILD_GENERAL1_2XLARGE" (824 GB).
def _build_codebuild():
    return build_image(
        codebuild=codebuild, s3=s3, iam=iam, logs=logs,
        account_id=ACCOUNT, region=REGION, bucket=BUCKET,
        container_dir=PROJECT / "container",
        ecr_repo=ECR_REPO, image_tag=IMAGE_TAG,
        compute_type="BUILD_GENERAL1_LARGE",
    )


print(state.cached("image.pushed", _build_codebuild))

In [ ]:
# Alternative: build locally. Only use this if the check above said there is
# enough room on Docker's data-root.
def _build_local():
    if not have_docker():
        raise RuntimeError("no Docker daemon here - use the CodeBuild cell above")
    cdir = PROJECT / "container"
    for f in ("build_and_push.sh", "serve"):
        (cdir / f).chmod(0o755)
    rc = subprocess.call(["./build_and_push.sh", "-r", REGION, "-n", ECR_REPO, "-t", IMAGE_TAG],
                         cwd=cdir)
    if rc != 0:
        raise RuntimeError("image build/push failed - see the output above")
    return {"image_uri": IMAGE_URI, "built_at": dt.datetime.now(dt.timezone.utc).isoformat()}


# print(state.cached("image.pushed", _build_local))
print("uncomment the line above to build locally instead of in CodeBuild")

In [ ]:
# Verify the image is really in ECR. Run this instead of the cell above if you
# built elsewhere (CodeBuild / EC2).
d = ecr.describe_images(repositoryName=ECR_REPO, imageIds=[{"imageTag": IMAGE_TAG}])["imageDetails"][0]
print(f"{IMAGE_URI}\n  pushed {d['imagePushedAt']}  {d['imageSizeInBytes'] / 1e9:.1f} GB compressed")
state.put("image.pushed", {"image_uri": IMAGE_URI, "pushed_at": str(d["imagePushedAt"])})

---
## 7. Deploy

**The meter starts here: $63.296/hour for as long as the endpoint exists.**

Async rather than real-time because a 5-second clip at 50 steps takes minutes on 4x H100 and a
real-time endpoint caps a response at 60 seconds. Async reads the request from S3, works for up to
`InvocationTimeoutSeconds` (3600 max), and writes the response to S3.

Deploy-time choices, all in `h3/deploy.py`:

- **Uncompressed model data** (`CompressionType: None` over a prefix) so SageMaker mirrors it into
  `/opt/ml/model` verbatim. Iterating on the container never re-archives 288 GB.
- **Both timeouts at the 3600 s maximum** — one covers a 288 GB download, the other two 4-GPU model
  loads after it.
- **No `VolumeSizeInGB`**: p5 has local NVMe and SageMaker rejects an explicit EBS size for families
  that do.
- **Delete before recreate**: pointing an existing name at a new model does not reliably replace a
  crash-looping container.
- **`MaxConcurrentInvocationsPerInstance: 1`** — both backends already use every GPU.
- **`H3_WARMUP_ON_START=1`**: `/ping` stays 503 until each backend completes one tiny real
  generation, so `InService` means "can actually generate".
- **SNS optional**: set `SNS_SUCCESS_TOPIC` for completion events. `IncludeInferenceResponseIn` is
  deliberately never set.

In [ ]:
spec = make_spec()
print(spec.describe())
print("\nenvironment:")
for k, v in spec.environment().items():
    print(f"  {k}={v}")
print("\nAsyncInferenceConfig:")
print(json.dumps(spec.async_inference_config(max_concurrent_invocations=1), indent=1))

In [ ]:
# Create the endpoint, unless it already exists. Safe to re-run.
status, reason = endpoint_status()
print(f"current status: {status}" + (f" ({reason})" if reason else ""))

FORCE_REDEPLOY = False   # True -> delete and recreate even if InService

if status == "InService" and not FORCE_REDEPLOY:
    print("already InService - skipping. Set FORCE_REDEPLOY = True to replace it.")
elif status in ("Creating", "Updating") and not FORCE_REDEPLOY:
    print("already coming up - skip to the wait cell.")
else:
    arn = deploy(sm, make_spec(), replace=True)
    state.put("endpoint", {"name": ENDPOINT_NAME, "arn": arn,
                           "created_at": dt.datetime.now(dt.timezone.utc).isoformat()})
    print(arn)

In [ ]:
# Wait. 20-40 min: image pull, 288 GB download, two model loads, two warmups.
# Returns immediately if already InService, so re-running is free.
if endpoint_status()[0] == "InService":
    print("InService")
else:
    st = wait_for_endpoint(sm, logs, ENDPOINT_NAME, timeout_s=4800, poll_s=30)
    assert st == "InService", f"endpoint is {st}"

### If it does not come up

```python
sm.describe_endpoint(EndpointName=ENDPOINT_NAME)["EndpointStatus"]     # Creating = still working
sm.describe_endpoint(EndpointName=ENDPOINT_NAME).get("FailureReason")  # the real answer on Failed
```

- **`Creating` for a long time** is normal. Do not intervene before ~40 minutes.
- **No log group** means the container never booted: the failure was in SageMaker's managed bootstrap
  (image pull, model download, capacity) — upstream of this code.
- **`[shim ...]` lines present** means the shim is running. Follow its sequence:
  `staged weights found` → `HF cache primed` → `[fl2va] HTTP up` → `warmup complete` → `READY`.
  Whichever step is missing is the one that failed.
- **`[fl2va]` / `[ref2va]` lines** are the SGLang subprocesses. A CUDA OOM there means the topology
  does not fit — set `PERFORMANCE_MODE = "memory"` or `VARIANTS = "fl2va"` and redeploy.

In [ ]:
tail_logs(minutes=60, limit=80)

In [ ]:
# Just the milestones.
tail_logs(minutes=90, limit=60, pattern='?"[shim" ?READY ?"BOOT FAILED" ?Traceback ?"out of memory"')

---
## 8. Demo 1 — text to video + audio (`t2va`)

The prompt is not a caption. `h3/prompts.py` builds the H3-Context-IR shape the checkpoint was
trained on:

```
integrated_multimodal_description: [Shot 1] ...  [Shot 2] At 00:03.200, the camera cuts to ...

overall_soundscape: ...

non_diegetic_music: ...
```

Rules it enforces or encodes, from `VIDEO_PROMPT_WRITING_GUIDE_base_en.md`:

- Shot 1 carries **no** timestamp; later shots need strictly increasing cut times inside the
  duration. `Shot(..., cut_time=)` validates this, so a bad cut fails locally instead of producing a
  quietly wrong video.
- Camera motion is prose from a fixed vocabulary — *motion type + amplitude + speed*
  (`h3.prompts.CAMERA_MOTION`) — not stacked labels.
- `overall_soundscape` is ambience, physical action sound and non-verbal human sound **only**.
  Dialogue, singing and diegetic music belong in the description; repeating them here is the most
  common mistake.
- `non_diegetic_music` is score the characters cannot hear: instrumentation, tempo, rhythm, dynamics.
  Not mood adjectives. `"N/A"` when there is none.

`flow_shift=12.0` / `audio_flow_shift=3.0` are the released schedules (video and audio latents step
down two different schedules inside one transformer call). `num_inference_steps` counts sigma grid
points including the terminal zero.

`extract=True` makes the endpoint also write this clip's first frame, last frame and soundtrack to
S3. Sections 9 and 10 chain off those, so nothing is ever downloaded.

In [ ]:
# Self-contained: builds the prompt, shows the rendered H3-Context-IR shape, then
# generates (or reuses) the clip.
t2va_prompt = BasePrompt(
    mode="t2va",
    duration_s=5.0,
    shots=[
        Shot(
            "Cinematic, live-action, a medium shot frames a cluttered workshop bench at night, lit "
            "by a single warm desk lamp. A brass pocket watch lies open on the oak surface beside a "
            "scatter of tiny gears and a pair of steel tweezers. The camera pushes in with small "
            "amplitude at slow speed toward the watch face as the second hand begins to sweep, and "
            "the exposed escapement rocks visibly back and forth. Fine dust drifts through the lamp "
            "beam above the bench."
        ),
        Shot(
            "an extreme close-up of the mainspring barrel, where a single gear tooth slips and the "
            "wheel train spins free for a moment before catching again.",
            cut_time=3.2,
        ),
    ],
    soundscape=(
        "A steady mechanical ticking carries throughout, close and dry against a quiet room tone. "
        "A faint metallic whirr rises as the wheel train spins free, followed by a small sharp click "
        "as it catches."
    ),
    music="Sparse vibraphone notes at a slow tempo over a sustained low cello drone that fades near the end.",
)
print(t2va_prompt.render())
print("\n" + "-" * 78 + "\n")

r_t2va = demo(
    "t2va",
    prompt=t2va_prompt,
    aspect_ratio="16:9",
    short_edge=768,
    seed=1101,
    num_inference_steps=50,
    flow_shift=12.0,
    audio_flow_shift=3.0,
    extract=True,          # first frame, last frame and soundtrack -> S3
)
r_t2va.show()
print("\nartifacts:")
for k, v in sorted(r_t2va.artifacts.items()):
    print(f"  {k:<12} {v}")

In [ ]:
# The extracted stills, straight from S3.
for which, label in (("first_frame", "frame 0"), ("last_frame", "final frame")):
    try:
        show_image(artifact("t2va", which), width=360, label=label)
    except KeyError as exc:
        print(exc)

### Demo 1b — `t2va`, a satirical second sample

Same wire task, very different register — a piece of junk-food-addiction satire. Things this
sample demonstrates that the watchmaker clip does not:

- **Comedy is timing, and timing is cut times.** The joke has a setup (the ritual unwrap), a
  beat of restraint (the stare, the breathing), and a punchline (the lunge). Each lives in its
  own shot with an explicit cut time, because H3 will pace what you time, not what you imply.
- **The satire is carried by the *non-diegetic* score.** The visuals play it straight; the
  music scores the burger like the climactic kill of a great hunt. Putting the irony in the
  music track keeps every visual sentence concrete, which is what the writing guide demands.
- **Describe, don't name.** The creature is written as a physical description (mandibles,
  dreadlock-like tendrils, warrior plates, shoulder cannon) rather than a franchise name —
  concrete appearance is what the checkpoint was trained on, and it generalises where a proper
  noun may not.


In [ ]:
# Satire sample: an apex hunter meets a triple cheeseburger - and no ketchup.
# Six timed beats in H3's maximum 15 s: ritual unwrap, restrained stare, the bite,
# frozen disbelief, forensic inspection, and armed condiment rage. The one spoken
# line goes through dialogue() so it lands as voiced speech, not set dressing.
junkfood_prompt = BasePrompt(
    mode="t2va",
    duration_s=15.0,
    shots=[
        Shot(
            "Cinematic, live-action, a medium static shot frames a hulking, heavyset alien "
            "hunter crammed into a small booth of a garish fast-food coffee shop. His broad, "
            "bulging belly presses against the formica table and the netted under-armour "
            "strains over it; battered metal warrior plates cover his shoulders, one carrying "
            "a small dormant shoulder cannon, and long black dreadlock-like tendrils hang "
            "around his crab-like mandibled face. Under flat fluorescent light, a "
            "grease-spotted paper parcel sits on a red plastic tray in front of him next to a "
            "paper coffee cup. He glances slowly left and right, then leans toward the parcel "
            "as the camera pushes in with small amplitude at slow speed."
        ),
        Shot(
            "a close-up of the tray as his clawed, gauntleted hands unwrap the paper parcel "
            "with exaggerated ceremonial care, fold by fold. Inside sits a towering triple "
            "cheeseburger, three stacked patties with melted cheese sliding down the sides, "
            "and a thin curl of steam rises into the fluorescent light.",
            cut_time=2.5,
        ),
        Shot(
            "a close-up of his mandibled face staring down at the burger. His mandibles "
            "spread slowly apart and his chest heaves with deepening breaths, then he "
            "snatches the burger fast with both hands and drives it into his open mandibles, "
            "taking one enormous bite that pulls long ropes of melted cheese.",
            cut_time=5.0,
        ),
        Shot(
            "the same close framing as he freezes mid-chew. His mandibles stop moving, his "
            "eyes widen, and he stares straight ahead in disbelief while a strand of cheese "
            "hangs from his jaw.",
            cut_time=7.5,
        ),
        Shot(
            "a medium shot as he holds the bitten burger up into the fluorescent light and "
            "turns it slowly in his claws, inspecting it from different angles. He tilts it "
            "left, then right, lifts the top bun with one claw and peers underneath like a "
            "forensic examiner, and his tendrils quiver.",
            cut_time=9.5,
        ),
        Shot(
            "a medium shot as he surges up out of the booth in fury, his belly knocking the "
            "tray, and draws a glowing high-tech blade from behind his shoulder in one motion. "
            "He slashes the blade down through the formica table, splitting it cleanly in two "
            "so both halves collapse and the coffee cup slides to the floor, and "
            + dialogue("the alien hunter", "S1", "No ketchup!", verb="roars")
            + " with his mandibles flared wide.",
            cut_time=12.0,
        ),
    ],
    soundscape=(
        "A flat fluorescent hum and a distant deep-fryer sizzle sit under thin, cheerful cafe "
        "muzak from a ceiling speaker. Paper crinkles fold by fold as the parcel opens. Low "
        "guttural clicks deepen into heavy rasping breaths, then a single enormous wet bite "
        "lands. The chewing stops dead into near-silence with only the muzak continuing. Soft "
        "leathery creaks accompany the slow turning of the burger. Then vinyl squeals as he "
        "rises, an energised metallic shring rings as the blade draws, one clean shearing "
        "crack splits the table, the tray and cup clatter across the floor, and his roaring "
        "voice tears through the room while the cheerful muzak plays on, oblivious."
    ),
    music=(
        "A solemn epic orchestral score with low brass, distant choir and pounding drums "
        "builds through the unwrapping and peaks exactly on the bite, then cuts out dead "
        "during the frozen disbelief. It creeps back as a single tense staccato string "
        "figure during the inspection, erupts into furious full-orchestra hits as the blade "
        "draws and falls, and stops hard on the final roar."
    ),
)
print(junkfood_prompt.render())
print("\n" + "-" * 78 + "\n")

r_junkfood = demo(
    "t2va-junkfood-v2",     # new state key: the 10 s no-dialogue cut stays cached as t2va-junkfood
    prompt=junkfood_prompt,
    aspect_ratio="16:9",
    short_edge=768,
    seed=7707,
    num_inference_steps=50,
    flow_shift=12.0,
    audio_flow_shift=3.0,
    extract=True,
)
r_junkfood.show()


In [ ]:
# Listen to the scene's background noise on its own. Nothing extra to generate:
# H3 denoises audio and video JOINTLY, so this soundscape was produced in the same
# pass as the pixels, and extract=True had the endpoint strip it into a standalone
# 32 kHz stereo WAV in S3 next to the MP4. Streamed via a presigned URL, same as
# the clips - no bytes land on this instance.
def show_audio(uri, label=""):
    display(HTML(
        f"<p style='font:600 12px sans-serif;margin:6px 0 2px'>{label or uri.rsplit('/', 1)[-1]}</p>"
        f'<audio controls preload="none" src="{presign(uri)}" style="width:420px"></audio>'
    ))

show_audio(artifact("t2va-junkfood-v2", "audio"),
           label="junk-food satire - generated soundscape (fryer, muzak, breathing, the bite, the blade, the roar)")


---
## 9. Demo 2 — keyframe conditioning (`fl2va`)

One wire task, three prompt shapes. The writing guide treats them as distinct because the alignment
instruction on the first line differs and the description has to be written in a different direction:

| Prompt `mode` | Keyframes | Instruction says | Write the description as |
|---|---|---|---|
| `i2va` | first frame | Picture 1 is the frame at 0.00 s | anchor → action onset → development → result |
| `l2va` | last frame | Picture 1 lands at the end | plausible earlier state → transition → convergence |
| `fl2va` | both | Picture 1 at 0.00 s, Picture 2 at the end | first-frame state → intermediate changes → narrowing differences → last-frame state |

`BasePrompt.wire_task` resolves all three to `"fl2va"`, so the prompt and the payload cannot
disagree.

- **`aspect_ratio="auto"`** — for keyframe tasks the canvas follows the first keyframe. Forcing a
  mismatched ratio makes the model crop or pad, which shows up as a re-rendered, subtly wrong subject.
- **`frame_index=0` marks the opening frame.** Omit it and the image is the *closing* frame — exactly
  what `l2va` wants.

All three read their keyframes from S3 (the artifacts extracted in section 8).

### 9a. First frame → video (`i2va`)

In [ ]:
r_i2va = demo(
    "i2va",
    prompt=BasePrompt(
        mode="i2va", duration_s=5.0,
        shots=[Shot(
            "Live-action, cinematic. The frame opens on the workshop bench exactly as shown, the open "
            "pocket watch under the warm lamp with gears scattered around it. A hand in a grey sleeve "
            "enters from the right holding steel tweezers, lowers them into the watch movement, and "
            "lifts one small brass gear clear of the plate. The camera holds a static shot as the gear "
            "rises into the lamp beam and catches the light."
        )],
        soundscape=(
            "Quiet room tone with a steady close ticking underneath. Fabric rustles as the sleeve "
            "moves, and a single faint metallic tap sounds as the tweezers touch the plate."
        ),
        music="N/A",
    ),
    conditions=[keyframe(artifact("t2va", "first_frame"), frame_index=0)],
    aspect_ratio="auto", seed=2202, num_inference_steps=50,
)
r_i2va.show()

### 9b. Last frame → video (`l2va`)

Same wire task, inverted description: infer a plausible earlier state and converge on the image.

In [ ]:
r_l2va = demo(
    "l2va",
    prompt=BasePrompt(
        mode="l2va", duration_s=5.0,
        shots=[Shot(
            "Live-action, cinematic. The shot begins on the darkened workshop bench with the desk lamp "
            "off and the pocket watch closed. The lamp flicks on and warm light spreads across the oak "
            "surface. A hand in a grey sleeve reaches in, thumbs the watch case open and withdraws, and "
            "the camera pushes in with small amplitude at slow speed as the composition settles into "
            "the final framing of the open movement under the lamp."
        )],
        soundscape=(
            "A soft click as the lamp switch throws, then a low electrical hum. Fabric rustles once, a "
            "small metallic snap as the case opens, and a steady close ticking rises and continues to "
            "the end."
        ),
        music="N/A",
    ),
    conditions=[keyframe(artifact("t2va", "last_frame"))],   # no frame_index -> closing frame
    aspect_ratio="auto", seed=3303, num_inference_steps=50,
)
r_l2va.show()

### 9c. First **and** last frame (`fl2va`)

With both endpoints pinned the model interpolates a continuous path. The guide is specific: prefer a
**single shot** unless cuts are explicitly wanted, and describe progressively narrowing differences so
the final shot lands on Picture 2.

In [ ]:
r_fl2va = demo(
    "fl2va",
    prompt=BasePrompt(
        mode="fl2va", duration_s=6.0,
        shots=[Shot(
            "Live-action, cinematic, a single continuous shot. The frame begins on the workshop bench "
            "as shown in the first image, the open pocket watch under the warm lamp. The camera arcs "
            "slowly to the left around the bench while the lamp's warm light slides across the oak "
            "grain and the scattered gears throw moving shadows. Dust drifts steadily through the "
            "beam. As the arc completes, the composition settles and the framing, lighting and every "
            "object position converge on the final image."
        )],
        soundscape=(
            "A steady close ticking runs throughout over quiet room tone, with a faint low hum from "
            "the lamp that stays constant."
        ),
        music="A single sustained cello note that swells gently and holds.",
    ),
    conditions=[
        keyframe(artifact("t2va", "first_frame"), frame_index=0),
        keyframe(artifact("t2va", "last_frame")),
    ],
    aspect_ratio="auto", duration_s=6.0, seed=4404, num_inference_steps=50,
)
r_fl2va.show()

---
## 10. Demo 3 — omni-references (`ref2va`)

The other checkpoint partition, and where H3 stops being a text-to-video model. It takes an *ordered*
list of up to 9 images, 3 videos and 3 audio clips (12 total) and conditions on all of them jointly.

**Order is semantic.** A reference's position in `conditions` is what `<Picture 1>`, `<Video 1>`,
`<Audio 1>` in the prompt refer to, and it advances H3's shared audio/video rotary clock — so
re-ordering the same references is a genuinely different request.

`RefPrompt` emits the six sections the reference guide fixes, in order:

| Section | What it does |
|---|---|
| `subject_definitions` | binds each label to content and states its role |
| `summary` | `[task type + task type]` prefix, then one paragraph |
| `retention_analysis` | per reference: `fully_preserved` / `partially_preserved` / `transferred` / `partially_copy` / `reference` |
| `detailed_description` | the shot-by-shot body, citing labels |
| `overall_soundscape` | ambience and physical sound |
| `non_diegetic_music` | score only the audience hears |

`RefPrompt` cross-checks the labels: cite `<Video 3>` without defining it and construction fails
locally. An undefined label is a silent quality loss, not an error the model reports.

Two more constraints:

- References **do not bind the output geometry** (unlike keyframes). They are encoded at their own
  resolution and the canvas defaults to 16:9. `aspect_ratio="auto"` resolves to that 16:9 fallback for
  `ref2va` — it does *not* inherit a reference's shape.
- Audio can never be the only reference; it must accompany an image or video.

### 10a. Image reference — carry a subject into a new scene

In [ ]:
r_ref_img = demo(
    "ref2va-image",
    prompt=RefPrompt(
        duration_s=5.0,
        subject_definitions=[
            "<Picture 1> is a photograph of a brass pocket watch lying open on an oak workbench under "
            "a warm desk lamp.",
            "<Subject 1> is the brass pocket watch from <Picture 1>, with its open case, exposed gear "
            "train and worn engraved lid.",
        ],
        task_types=["reference generation"],
        summary=(
            "The target video places <Subject 1> in a new setting: the same pocket watch resting on a "
            "windowsill in cold daylight while rain runs down the glass behind it."
        ),
        retention_analysis=[
            "<Subject 1> (appears in [Shot 1]): fully_preserved - the brass case, engraved lid and "
            "exposed gear train keep the appearance shown in <Picture 1>.",
            "<Picture 1> (setting): transferred - only the watch is carried over; the workbench and "
            "warm lamp are replaced by a windowsill in cold daylight.",
        ],
        shots=[Shot(
            "The frame opens on a painted wooden windowsill in flat, cool daylight. <Subject 1> rests "
            "open on the sill, its brass catching the grey light, and behind it rain runs in thin "
            "streaks down the window glass. The camera pushes in with small amplitude at slow speed "
            "toward the open movement while the second hand sweeps and a rain drop crosses the pane "
            "directly behind the watch."
        )],
        soundscape=(
            "Steady rain taps against the glass with a low outdoor rumble underneath. A close, dry "
            "mechanical ticking continues throughout."
        ),
        music="A slow felt-piano figure repeating at a steady tempo, quiet and even.",
    ),
    conditions=[image_ref(artifact("t2va", "first_frame"))],
    seed=5505, num_inference_steps=50,
)
r_ref_img.show()

### 10b. Video reference — video-to-video editing

V2V is not a fourth task. It is `ref2va` with a `video` reference, served by the same `Ref2VA`
partition. `<Video N>` is reserved for **whole-video** relationships: editing a source, continuing
from its end, or borrowing its camera movement, cuts and rhythm. A person or object lifted out of a
reference video is still a `<Subject N>`.

The source is the clip from section 8, still in S3 — it is never downloaded.
`start_time_seconds` seeks into it; visual and audio streams are always sought together.

In [ ]:
r_ref_vid = demo(
    "ref2va-v2v",
    prompt=RefPrompt(
        duration_s=5.0,
        subject_definitions=[
            "<Video 1> is the source video for the target video edit, a close shot of an open brass "
            "pocket watch on an oak workbench under a warm desk lamp.",
            "<Subject 1> is the brass pocket watch in <Video 1>, with its open case and exposed gear "
            "train.",
        ],
        task_types=["video editing"],
        summary=(
            "The target video is an edited version of <Video 1>. The camera movement, framing and "
            "<Subject 1> are kept, while the warm tungsten lighting is regraded to cold blue "
            "moonlight and a slow drift of pale mist is added across the bench."
        ),
        retention_analysis=[
            "<Video 1> (source video editing): partially_preserved - the original framing, camera "
            "movement and timing are kept while the lighting and colour are changed.",
            "<Subject 1> (appears in [Shot 1]): fully_preserved - the watch keeps its case shape, "
            "engraving and visible movement parts.",
        ],
        shots=[Shot(
            "The shot begins from the source <Video 1>, framing <Subject 1> open on the oak "
            "workbench, and follows the same slow push in with small amplitude. The lighting is now "
            "cold blue moonlight raking in from the left rather than warm lamplight, and the oak "
            "grain reads grey instead of amber. Thin pale mist drifts steadily from right to left "
            "across the bench in front of the watch, and the brass throws a cool highlight as the "
            "second hand sweeps."
        )],
        soundscape=(
            "A close, dry mechanical ticking continues throughout. The room tone is colder and "
            "emptier, with a faint low draught moving under it."
        ),
        music="A single sustained synthetic pad note, quiet and unchanging.",
    ),
    conditions=[video_ref(state.require("demo.t2va", hint="section 8")["s3_uris"][0],
                          start_time_seconds=0.0)],
    seed=7707, num_inference_steps=50,
)
r_ref_vid.show()

### 10c. Audio reference — soundtrack reuse and voice timbre

`<Audio N>` covers copying an audio signal, referencing a music style, referencing a speaker's timbre
and delivery, or borrowing beat and rhythm. Reusing a soundtrack and driving lip-sync from a voice
clip are the same mechanism with different `retention_analysis` verbs (`partially_copy` vs
`reference`).

Dialogue goes inside `<d>[Language] ...</d>` with identity, speaker id and delivery **outside** the
tag — `h3.prompts.dialogue()` gets that right. Speaker ids (`S1`, `S2`, `S1,S2` when they speak
together) stay stable across shots; characters who never vocalise get no id.

This reuses the soundtrack H3 generated in section 8 — proof the audio really is a separate stream
you can feed back in. Section 10d is the lip-sync variant with your own voice clip.

In [ ]:
r_ref_aud = demo(
    "ref2va-audio",
    prompt=RefPrompt(
        duration_s=5.0,
        subject_definitions=[
            "<Picture 1> is a photograph of a brass pocket watch open on an oak workbench under a "
            "warm desk lamp.",
            "<Subject 1> is the brass pocket watch from <Picture 1>.",
            "<Audio 1> is a recording of close mechanical ticking and quiet room tone, reused as the "
            "target video's own diegetic sound.",
        ],
        task_types=["reference generation", "audio reuse"],
        summary=(
            "The target video shows <Subject 1> on a workbench while a hand winds its crown. The "
            "mechanical sound of <Audio 1> is reused as the video's diegetic audio and stays "
            "synchronised with the winding action."
        ),
        retention_analysis=[
            "<Subject 1> (appears in [Shot 1]): fully_preserved - the watch keeps the appearance "
            "shown in <Picture 1>.",
            "<Audio 1>: partially_copy - its mechanical ticking and room tone are reused as the "
            "target video's diegetic sound bed.",
        ],
        shots=[Shot(
            "The frame opens on <Subject 1> lying open on the oak workbench in warm lamplight. A hand "
            "in a grey sleeve enters from the right, takes the crown between finger and thumb and "
            "turns it in short repeated movements. The ticking of <Audio 1> continues under the action "
            "and a faint ratchet sound accompanies each turn. The camera holds a static shot and the "
            "exposed escapement rocks faster as the mainspring tightens."
        )],
        soundscape=(
            "The close mechanical ticking and quiet room tone of <Audio 1> continue throughout, with "
            "a small dry ratchet click on each turn of the crown and a light rustle of fabric."
        ),
        music="N/A",
    ),
    conditions=[image_ref(artifact("t2va", "first_frame")), audio_ref(artifact("t2va", "audio"))],
    seed=8808, num_inference_steps=50,
)
r_ref_aud.show()

### 10d. Lip-sync (bring your own portrait + voice)

Upload a portrait and a 2-15 s clip of one person speaking, then H3 animates the mouth to a new line
in that voice timbre. Upload straight to S3 — no local processing, nothing downloaded.

Leave the paths as `None` to skip this cell.

In [ ]:
PORTRAIT_LOCAL = None   # e.g. "me.jpg"  - uploaded to S3, not processed locally
SPEECH_LOCAL   = None   # e.g. "voice.wav" - 2-15 s of one person speaking
LINE           = "The escapement is the part everybody forgets."

if not (PORTRAIT_LOCAL and SPEECH_LOCAL):
    print("Set PORTRAIT_LOCAL and SPEECH_LOCAL to run the lip-sync demo.")
else:
    import mimetypes

    def upload(path):
        p = Path(path)
        key = f"{IO_KEY}/demo-inputs/{p.name}"
        s3.upload_file(str(p), BUCKET, key, ExtraArgs={
            "ContentType": mimetypes.guess_type(p.name)[0] or "application/octet-stream"})
        return f"s3://{BUCKET}/{key}"

    portrait_s3, speech_s3 = upload(PORTRAIT_LOCAL), upload(SPEECH_LOCAL)
    print("uploaded:", portrait_s3, speech_s3, sep="\n  ")

    r_lipsync = demo(
        "ref2va-lipsync",
        prompt=RefPrompt(
            duration_s=5.0,
            subject_definitions=[
                "<Picture 1> is a portrait photograph of a person facing the camera.",
                "<Subject 1> is the person in <Picture 1>.",
                "<Audio 1> is the voice-timbre reference for <Subject 1> (S1), containing spoken speech.",
            ],
            task_types=["reference generation", "audio reference"],
            summary=(
                "The target video animates <Subject 1> speaking a new line, with mouth movement "
                "synchronised to the speech and the voice timbre referenced from <Audio 1>."
            ),
            retention_analysis=[
                "<Subject 1> (appears in [Shot 1]): fully_preserved - identity, hair, clothing and "
                "lighting keep the appearance shown in <Picture 1>, with the mouth newly animated.",
                "<Audio 1>: reference - the target references its voice timbre and delivery for "
                "<Subject 1>'s spoken line.",
            ],
            shots=[Shot(
                "The frame opens on <Subject 1> in a medium close-up facing the camera, framed and lit "
                "as in <Picture 1>. The camera holds a static shot. <Subject 1> physically speaks, "
                "mouth movements syncing naturally to the dialogue, voice timbre referencing "
                "<Audio 1>. "
                + dialogue("The person facing the camera", "S1", LINE)
                + " As the line ends the lips settle closed and the gaze holds on the lens."
            )],
            soundscape="A quiet indoor room tone with no traffic or wind under the spoken line.",
            music="N/A",
        ),
        conditions=[image_ref(portrait_s3), audio_ref(speech_s3)],
        seed=8809, num_inference_steps=50,
    )
    r_lipsync.show()

---
## 11. Demo 4 — the request knobs

Each sub-demo is cached independently, so you can run just the one you care about.

### 11a. Aspect ratio and duration

`target` resolves the aligned canvas and frame count. Keep the short edge at 768 for the released
quality recipe; the aspect ratio is free (21:9, 16:9, 4:3, 1:1, 3:4, 9:16 and more). Duration is
4-15 s inclusive — the frame grid is `17n + 5` at 24 fps, so the server snaps up to the next
decodable count.

Four clips at roughly 3-6 minutes each is 15-25 p5-minutes, about $15-25. They submit together but
the endpoint runs them one at a time (`MaxConcurrentInvocationsPerInstance` is 1), so this shortens
*your* wait, not the GPU time.

In [ ]:
# tide() comes from the bootstrap, so each sub-demo below stands alone.
for name, ar, dur in [("ar-16x9", "16:9", 4.0), ("ar-9x16", "9:16", 4.0),
                      ("ar-21x9", "21:9", 4.0), ("dur-10s", "16:9", 10.0)]:
    res = demo(name, prompt=tide(dur), aspect_ratio=ar, duration_s=dur,
               seed=9109, num_inference_steps=50)
    res.show(width=420)

### 11b. Several outputs per prompt

`num_outputs_per_prompt` (or OpenAI's `n`) takes 1-10. A scalar seed expands deterministically as
`seed + output_index`, so the variants get different noise rather than being identical. Same-prompt
fan-out reuses the text conditioning, but denoise and decode stay sequential, so wall time scales
roughly with the count.

In [ ]:
r_multi = demo("two-outputs", prompt=tide(4.0), aspect_ratio="16:9", duration_s=4.0,
               seed=1200, num_inference_steps=50, num_outputs=2)
r_multi.show(width=420)

### 11c. Determinism

A request draws its conditioning noise, then the video noise, then the audio noise, all from one
generator. Same seed and same request means the same video *and* the same soundtrack. Worth verifying:
a silently non-deterministic pipeline makes every later A/B comparison meaningless.

Compared by S3 `ETag`, so nothing is downloaded.

In [ ]:
a = demo("determinism-a", prompt=tide(4.0), aspect_ratio="16:9", duration_s=4.0,
         seed=777, num_inference_steps=50)
b = demo("determinism-b", prompt=tide(4.0), aspect_ratio="16:9", duration_s=4.0,
         seed=777, num_inference_steps=50)


def etag(uri):
    p = urllib.parse.urlparse(uri)
    h = s3.head_object(Bucket=p.netloc, Key=p.path.lstrip("/"))
    return h["ETag"].strip('"'), h["ContentLength"]


ta, tb = etag(a.s3_uri), etag(b.s3_uri)
print(f"a: etag={ta[0]}  {ta[1]:,} bytes")
print(f"b: etag={tb[0]}  {tb[1]:,} bytes")
print("identical:", ta == tb)

### 11d. Speed levers

The biggest lever is **canvas size**, not step count: the packed sequence grows with area, so
960x544 runs roughly 2.3x faster per step than the trained 1344x768. Height and width only have to be
multiples of 32.

- `num_inference_steps` — 50 is the reference recipe; lower trades fidelity for time roughly linearly.
- `flow_shift` / `audio_flow_shift` — the video and audio schedules (12.0 / 3.0 as released). Changing
  them changes the sampling trajectory; an experiment, not a tuning dial.
- `quality: "high"` — SGLang's audited Cache-DiT path, ~1.40x faster at SSIM 0.931 / PSNR 28.16 dB
  against `lossless`. **Fail-closed to a 4x H200 deployment** with one exact workload, so on this
  4x H100 topology it is rejected before denoising. Documented so you know why, not to enable.

In [ ]:
for name, steps, edge in [("speed-50-768", 50, 768), ("speed-20-768", 20, 768),
                          ("speed-50-544", 50, 544)]:
    res = demo(name, prompt=tide(4.0), aspect_ratio="16:9", duration_s=4.0, seed=1301,
               num_inference_steps=steps, short_edge=edge)
    print(f"  {name:<14} steps={steps:<3} short_edge={edge}  generate {res.generate_s:6.1f}s")

---
## 12. Optional — the full 2K workflow

Two of H3's three modules are not open. To match a direct MiniMax API call you pair the locally served
H3-Base with their hosted endpoints:

1. **H3-Context-IR** rewrites free-form input into the structured representation. Sections 8-11
   hand-build that representation; this is the real thing.
2. **H3-Base** — the endpoint above, 768p.
3. **H3-Regenerate-2K** feeds the 768p result *plus the original context* back through H3. In-context
   regeneration, not super-resolution, which is why it can recover small text and fine detail an
   upscaler would have to invent.

Needs a token from [platform.minimax.io](https://platform.minimax.io) in `MINIMAX_API_TOKEN`. Skipped
without one, and it bills MiniMax credits, not AWS. Request and response shapes are in the model
card's "Full 2K-Workflow" section — check the current reference, since this is the one part of the
notebook that is not pinned.

In [ ]:
MINIMAX_API_TOKEN = ""            # <-- paste a MiniMax platform token to enable
MINIMAX_API_BASE = "https://api.minimax.io"

if not MINIMAX_API_TOKEN:
    print("MINIMAX_API_TOKEN not set - skipping the hosted Context-IR and 2K stages.")
    print("Everything above runs on your own endpoint and needs no MiniMax account.")
else:
    import base64 as _b64
    import urllib.request

    def minimax_post(path, body):
        req = urllib.request.Request(
            MINIMAX_API_BASE.rstrip("/") + path, data=json.dumps(body).encode(),
            headers={"Authorization": f"Bearer {MINIMAX_API_TOKEN}",
                     "Content-Type": "application/json"}, method="POST")
        with urllib.request.urlopen(req, timeout=300) as resp:
            return json.loads(resp.read().decode())

    ir = state.cached("minimax.context_ir", lambda: minimax_post(
        "/v1/video_generation/h3_context_ir",
        {"model": "MiniMax-H3",
         "prompt": "A watchmaker winds an antique pocket watch by lamplight; we hear the escapement "
                   "speed up. Cinematic, 10 seconds, 16:9.",
         "duration": 10, "ratio": "16:9"}))
    context_prompt = ir["task"]["content"]["prompt"]
    print(context_prompt[:1200], "...\n")

    r_ir = demo("context-ir", prompt=context_prompt, task="t2va", duration_s=10.0,
                aspect_ratio="16:9", seed=0, num_inference_steps=50)
    r_ir.show()

    # Regenerate-2K wants the video itself, so this is the one place bytes move
    # through the notebook. They go straight from S3 into the request body.
    p = urllib.parse.urlparse(r_ir.s3_uri)
    video_bytes = s3.get_object(Bucket=p.netloc, Key=p.path.lstrip("/"))["Body"].read()
    two_k = minimax_post("/v1/video_generation/regeneration", {
        "model": "MiniMax-H3",
        "base_video": "data:video/mp4;base64," + _b64.b64encode(video_bytes).decode(),
        "prompt": context_prompt})
    print(json.dumps(two_k, indent=1)[:1000])

---
## 13. Operations and teardown

**Billing.** An async endpoint with `InitialInstanceCount: 1` bills for the instance the whole time it
exists — $63.296/hour, about $1519/day. Delete it when you are not using it.

Async endpoints *can* scale to zero (`register_scalable_target` on
`SageMakerVariantInvocationsPerInstance` with `MinCapacity: 0`), but scaling back up re-downloads
288 GB and reloads two models — the full 20-40 minute cold start. For demo use, delete and redeploy
costs the same and has fewer failure modes.

**Iterating on the shim** does not touch the weights: rebuild and push (a few minutes, only the last
layers change), then re-run section 7 with `FORCE_REDEPLOY = True`. The 288 GB in S3 stays put, and
so does every generated clip.

In [ ]:
# Where the money went, and what is on record.
status, reason = endpoint_status()
print(f"endpoint {ENDPOINT_NAME}: {status}")
if status not in ("NotFound",):
    d = sm.describe_endpoint(EndpointName=ENDPOINT_NAME)
    age_h = (dt.datetime.now(dt.timezone.utc) - d["CreationTime"]).total_seconds() / 3600
    print(f"  up {age_h:.2f} h  ->  approx ${age_h * 63.296:,.2f} at $63.296/h")

for label, key in (("weights", WEIGHTS_KEY + "/"),
                   ("generated clips", IO_KEY + "/generated/"),
                   ("requests+responses", IO_KEY + "/")):
    n, total = s3_size(key)
    print(f"{label:<20} {n:>6} objects  {total / 1e9:8.1f} GB")

print()
print(state.summary())

In [ ]:
# Teardown: endpoint, config, model, dedicated role. Weights and clips stay in S3,
# and so does the state file, so a redeploy skips sections 5 and 6 and every demo
# you have already paid for is still viewable.
delete_endpoint(sm, ENDPOINT_NAME, wait=True)
delete_endpoint_role(iam, ENDPOINT_ROLE)
state.drop("role.arn")
state.drop("endpoint")
print("\nkept in S3:")
print(f"  {WEIGHTS_PREFIX}   288 GB, about $6.60/month")
print(f"  {IO_PREFIX}        clips, requests, responses, notebook state")

In [ ]:
# Optional: delete the staged weights too (288 GB). Everything else is small.
DELETE_WEIGHTS = False
if DELETE_WEIGHTS:
    n = 0
    for page in s3.get_paginator("list_objects_v2").paginate(Bucket=BUCKET, Prefix=WEIGHTS_KEY + "/"):
        batch = [{"Key": o["Key"]} for o in page.get("Contents", [])]
        if batch:
            s3.delete_objects(Bucket=BUCKET, Delete={"Objects": batch})
            n += len(batch)
    state.drop("weights.staged")
    print(f"deleted {n} objects")
else:
    print("DELETE_WEIGHTS is False - weights kept, so a redeploy skips section 5")

---
## Appendix — things that will bite you

**Serving**

| Symptom | Cause | Fix |
|---|---|---|
| Endpoint `Failed`, no log group | Container never booted: image pull, model download or capacity | `FailureReason`; check the image exists and the role can read the weights prefix |
| `AccessDenied` on `create_model` | Deploying principal lacks `iam:PassRole` on the endpoint role | Grant it scoped to `minimax-h3-endpoint-role`; the message never names PassRole |
| `The provided role '...' is invalid` on deploy | A `Condition` in the role policy — SageMaker validates with an empty request context, so `s3:prefix` / `aws:SourceAccount` conditions evaluate false | Re-run the role repair cell in section 4, then `diagnose_endpoint_role()`. Scope by resource ARN, never by Condition |
| `/ping` 503 forever | A backend never became healthy | Read the `[fl2va]` / `[ref2va]` log lines; the SGLang traceback is there |
| CUDA OOM on a long clip | 15 s at 768p roughly triples the packed sequence vs the benchmarked 5 s | `PERFORMANCE_MODE="memory"`, or `VARIANTS="fl2va"`, or shorten the clip |
| `task 'ref2va' needs...` 400 | This endpoint only loaded `fl2va` | `VARIANTS="fl2va,ref2va"` and redeploy |
| `docker build`: no space left | Docker's data-root is on the notebook instance's fixed ~100 GB root volume | Use the CodeBuild cell, or `scripts/notebook-lifecycle-on-start.sh` |
| `RuntimeError: no Docker daemon here` | SageMaker Studio has none unless the domain sets `EnableDockerAccess` | Use the CodeBuild cell in section 6 |
| CodeBuild fails in `build` on disk | 128 GB was not enough | Re-run with `compute_type="BUILD_GENERAL1_2XLARGE"` (824 GB) |
| `Client.get() got an unexpected keyword argument 'stream'` | An older `stage_weights.py` used `huggingface_hub.utils.get_session()`, which returns `requests.Session` on 0.x and `httpx.Client` on 1.x | Fixed: that path is stdlib `urllib` now and has no opinion about which HTTP library is installed |

**Model behaviour**

- **`quality: "high"` is fail-closed** to one exact workload on 4x H200. Rejected before denoising
  anywhere else.
- **No negative prompt, no guidance scale.** CFG-distilled: guidance is in the weights, one forward
  pass per step. CFG parallelism is rejected rather than silently duplicating the positive branch.
- **`torch.compile` changes the numbers.** SGLang keeps the DiT eager under `--performance-mode
  speed`. Never use a compiled run as ground truth.
- **`aspect_ratio: "auto"` means different things per task.** Keyframe tasks take the canvas from the
  first keyframe; `ref2va` resolves to H3's 16:9 fallback and does *not* inherit a reference's
  geometry.
- **Reference order is semantic.** It binds `<Picture N>` / `<Video N>` / `<Audio N>` and advances the
  shared audio/video rotary clock.
- **Audio is never a sole reference.**
- **Ring attention is incompatible** with H3's packed multi-segment attention. Ulysses only.
- **Sparse attention is not in this release**, so long clips pay full attention cost.

**Prompting** (from the two official guides)

- Shot 1 takes no timestamp; later shots need strictly increasing cut times inside the duration.
- Only the language tag and the verbatim words go inside `<d>...</d>`. Identity, speaker id, action and
  delivery stay outside.
- Voiceover needs the exact phrase `says in an off-screen voiceover`, immediately followed by a
  statement that the on-screen character's lips stay closed — otherwise a mouth gets animated to match
  narration.
- `overall_soundscape` must not repeat dialogue, singing or diegetic music.
- `non_diegetic_music` describes instrumentation, tempo, rhythm, dynamics — not mood.
- On-screen text goes in double quotes, verbatim and untranslated.
- Reference labels must be defined in `subject_definitions` before use. `RefPrompt` enforces this.

**References**

- [Model card](https://huggingface.co/MiniMaxAI/MiniMax-H3)
- [SGLang H3 cookbook](https://docs.sglang.io/cookbook/diffusion/MiniMax/MiniMax-H3) — topologies, benchmarks, request schema
- [diffusers pipeline docs](https://github.com/huggingface/diffusers/blob/minimax-h3/docs/source/en/api/pipelines/minimax_h3.md)
- `docs/VIDEO_PROMPT_WRITING_GUIDE_base_en.md` and `..._ref_en.md` in the model repo